In [ ]:
import bokeh.plotting as bpl
import cv2
import glob
import logging
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
from datetime import datetime

try:
    cv2.setNumThreads(0)
except():
    pass

try:
    if __IPYTHON__:
        # this is used for debugging purposes only. allows to reload classes
        # when changed
        get_ipython().magic('load_ext autoreload')
        get_ipython().magic('autoreload 2')
except NameError:
    pass
import warnings
warnings.filterwarnings("ignore", message=r"Passing", category=FutureWarning)
import caiman as cm
from caiman.motion_correction import MotionCorrect
from caiman.source_extraction.cnmf import cnmf as cnmf
from caiman.source_extraction.cnmf import params as params
from caiman.utils.visualization import plot_contours, view_patches

from motion_correction_functions import load_mc
from scipy.sparse import coo_matrix

1) set inital file path & parameters

In [ ]:
path ='/projectnb/sramirezlab/amonast/LOU/GCaMP/160R/TSeries-09062024-096' # Path to TSeries
base_dir ='/projectnb/sramirezlab/amonast/LOU' #experiment directory

#filter with identifier string & check filenames 
fnames = [os.path.join(path,f) for f in os.listdir(path) if ('mmap' in f)]
fnames.sort()
#fnames=[fnames[1]]
print(fnames[0])

#set some parameters
display_movie=False #true if want to watch raw and motion corrected videos with opencv

plotting=True #True for plotting rois during processing
save_results=True #save cnmf object and parameters in pkl files

load_prev_mc = False #True if loading a mc object via pkl file 
pw_rigid=True # True for non-rigid motion correction

2) Get animal & session name, make output directories \
this requires folder structure '../base_dir/animal/GCaMP/TSeries/'

In [ ]:
animal_path =os.path.split(path)[0]
animal=path.split(os.path.sep)[-2]
TSer = path.split(os.path.sep)[-1]

mc_path = os.path.join(animal_path,'caiman_output','motion_correct')
if not os.path.exists(mc_path):    
    os.makedirs(mc_path)
    
output_path = os.path.join(animal_path,'caiman_output','cnmf')
if not os.path.exists(output_path): 
    os.makedirs(output_path)

print('animal id: '+ animal)
print('TSeries: ' + TSer)
print('motion correction save path: '+ mc_path)
print('cnmf save path: '+ output_path)

3. Set caiman parameters

In [ ]:
fr = 30                             # imaging rate in frames per second
decay_time = 0.4                    # length of a typical transient in seconds

# parameters for source extraction and deconvolution
seedCNMF=True
p = 2                       # order of the autoregressive system
gnb = 2                     # number of global background components
merge_thr = 0.85            # merging threshold, max correlation allowed
rf = 25                     # half-size of the patches in pixels. e.g., if rf=25, patches are 50x50
stride_cnmf = 6             # amount of overlap between the patches in pixels
K = 7                       # number of components per patch
gSig = [5,5]               # expected half size of neurons in pixels
method_init = 'greedy_roi'  # initialization method (if analyzing dendritic data using 'sparse_nmf')
ssub = 1                    # spatial subsampling during initialization
tsub = 2                    # temporal subsampling during intialization
only_init=False if seedCNMF else True
rf = None if seedCNMF else rf

# parameters for component evaluation
min_SNR = 2.0            # signal to noise ratio for accepting a component
rval_thr = 0.85              # space correlation threshold for accepting a component
cnn_thr = 0.99              # threshold for CNN based classifier
cnn_lowest = 0.1 # neurons with cnn probability lower than this value are rejected

opts_dict = {'fnames': fnames,
                'fr': fr,
                'decay_time': decay_time,
                'pw_rigid': pw_rigid,
                'p': p,
                'nb': gnb,
                'rf': rf,
                'K': K,
                'stride': stride_cnmf,
                'method_init': method_init,
                'rolling_sum': True,
                'only_init': only_init,
                'ssub': ssub,
                'tsub': tsub,
                'merge_thr': merge_thr,
                'min_SNR': min_SNR,
                'rval_thr': rval_thr,
                'use_cnn': True,
                'min_cnn_thr': cnn_thr,
                'cnn_lowest': cnn_lowest}

opts = params.CNMFParams(params_dict=opts_dict)
opts.data['path']=path
opts.data['animal']=animal
opts.data['TSeries']=TSer

3) load motion correction params if loading motion corrected files (optional)

In [ ]:
if load_prev_mc:
    opts_file = [os.path.join(mc_path,f) for f in os.listdir(mc_path) if (os.path.split(path)[-1] in f)&('mc_opts.pkl' in f)]
    mc_file = [os.path.join(mc_path,f) for f in os.listdir(mc_path) if (os.path.split(path)[-1] in f)&('mc.pkl' in f)]
    mc_og = load_mc(mc_file[0])

4) start cluster

In [ ]:
#%% Setup Cluster
if not load_prev_mc:
    if 'dview' in locals():
        cm.stop_server(dview=dview)
    c, dview, n_processes = cm.cluster.setup_cluster(
        backend='local', n_processes=None, single_thread=False)
    print('Starting cluster')
    mc = MotionCorrect(fnames, dview=dview, **opts.get_group('motion'))
    border_to_0 = 0 if mc.border_nan == 'copy' else mc.border_to_0

5) Write mmap files

In [ ]:
#%%Memory mapping
# The cell below  memory maps the file in order `'C'` and then loads the new memory mapped file.The saved files  from
# motion correction are memory mapped files stored in 'F' order their paths are stored ibn mc.mmap_file

# %% MEMORY MAPPING
# if C order memmap file exists already use it 
try:
    fname_new = [os.path.join(path,f) for f in os.listdir(path) if 'memmap__' in f][0]
    print('Loading memmap C file')
    print(fname_new)
   
#memory map the file in order 'C'
except:
    print('Writing mmap files from motion corrected files: '+ fnames[0])
    fname_new = cm.save_memmap(fnames, base_name='memmap_', order='C',
                            border_to_0=border_to_0, dview=dview)  # exclude borders

Yr, dims, T = cm.load_memmap(fname_new)
images = np.reshape(Yr.T, [T] + list(dims), order='F')  # load frames in python format (T x X x Y)
print(fname_new)

In [ ]:
Y_hat = cnm.estimates.A @ cnm.estimates.C  # Reconstructed signal

# Step 2: Compute residuals
Residuals = Yr - Y_hat  # Difference between observed and reconstructed data
Residuals_reshaped = Residuals.reshape((40000,256,256))


In [ ]:
Residuals_reshaped = Residuals.reshape((256,256,40000))


In [ ]:
Y_hat_reshaped=np.reshape(Y_hat.T, [T] + list(dims), order='F')

In [ ]:
Y_hat_reshaped.shape

In [ ]:
fig,ax=plt.subplots(ncols=2)
ax[0].imshow(Y_hat_reshaped.mean(axis=0))
ax[1].imshow(images.mean(axis=0))

In [ ]:
Y_hat_reshaped.shape

In [ ]:
recdata = cm.movie(Y_hat_reshaped)
recdata.play(fr=60,gain=1,q_min=0.1,q_max=99.6)

In [ ]:
resid = cm.movie(Residuals_reshaped.transpose(2,1,0))
resid.play(fr=60,gain=1,q_min=0.1,q_max=99.6)

In [ ]:
resid.save('/projectnb/sramirezlab/amonast/LOU/GCaMP/160R/TSeries-09062024-096/residual_movie2.tif')

In [ ]:
cnm2.estimates.b.shape

In [ ]:
np.unique(Ynew)

In [ ]:
plt.imshow(Cn)
Ynew=Y_hat_reshaped.mean(axis=0)
Ynew[Ynew==0]=np.nan
plt.imshow(Ynew,cmap='gray')

optional: play movie

In [ ]:
movie = cm.movie(images)
movie.play(fr=60, gain=1, magnification=1,offset=0,q_min=.1,q_max=99.5)

if seeding cnmf with cell masks

In [ ]:

if seedCNMF:
    seg_file = glob.glob(f"{path}/*{TSer}*seg.npy")
    print('masks for seeded cnmf from cellpose output: '+seg_file[0])
    seg=np.load(seg_file[0],allow_pickle=True).item()
    masks_2d=seg['masks']
    # labels=np.unique(masks_2d)
    # labels = labels[labels != 0]  # Remove background (label 0)

    # # Create an empty 3D array to hold individual masks
    # height, width = masks_2d.shape
    # n_masks=len(labels)
    # masks_3d = np.zeros((n_masks,height, width), dtype=np.uint8)

    # # Populate the 3D array with individual masks
    # for i, label in enumerate(labels):
    #     masks_3d[i,:,:] = (masks_2d == label).astype(np.uint8)

    def masks_to_sparse(masks_2d):
        """
        Convert a labeled 2D mask into a sparse matrix A.

        Args:
            masks_2d (np.ndarray): 2D array where each unique integer value represents an ROI.

        Returns:
            A (scipy.sparse.csc_matrix): Sparse matrix where columns correspond to ROIs and rows to pixels.
        """
        # Get the number of unique ROIs (excluding 0 if it's the background)
        unique_labels = np.unique(masks_2d)
        unique_labels = unique_labels[unique_labels > 0]  # Exclude background (label 0)
        num_components = len(unique_labels)

        # Initialize an empty dense matrix to store pixel-to-ROI mapping
        A = np.zeros((masks_2d.size, num_components), dtype=bool)

        # Loop over each unique ROI and create binary masks
        for i, label in enumerate(unique_labels):
            temp = (masks_2d == label)  # Binary mask for the current ROI
            A[:, i] = temp.flatten('F')  # Flatten in column-major order (Fortran-style)

        # Convert A to a sparse matrix for memory efficiency
        #A_sparse = csc_matrix(A)

        return A

    Aseed = masks_to_sparse(masks_2d)
    
    #plt.figure(); plt.imshow(np.reshape(A_csc.toarray().max(axis=1), dims, order='F'),origin='upper')

6) Run cnmf, calculate correlation image, display components

In [ ]:
print('Running CNMF')
# %% RUN CNMF ON PATCHES
# First extract spatial and temporal components on patches and combine them
# for this step deconvolution is turned off (p=0). If you want to have deconvolution within each patch change
# params.patch['p_patch'] to a nonzero value
Ain=Aseed if seedCNMF else None
cnm = cnmf.CNMF(n_processes, params=opts, dview=dview,Ain=Ain)
cnm = cnm.fit(images)

In [ ]:
###calculate correlation image
if dims == (512,512): # higher resolution, assume larger dataset, downsample for correlation image.
    rs_images = images.transpose(1, 2, 0)
    ds_images = rs_images[:,:,::2]
    Cn = cm.local_correlations(ds_images)
else:
    Cn = cm.local_correlations(images.transpose(1, 2, 0))
Cn[np.isnan(Cn)] = 0

### plot contours of found components
cnm.estimates.plot_contours(img=Cn)

### RE-RUN seeded CNMF on accepted patches to refine and perform deconvolution
cnm2 = cnm.refit(images, dview=dview)
cnm2.estimates.Cn = Cn # Update cnm2 object with Cn image

6) COMPONENT EVALUATION
   the components are evaluated in three ways:\
    a) the shape of each component must be correlated with the data\
    b) a minimum peak SNR is required over the length of a transient\
    c) each shape passes a CNN based classifier

In [ ]:
cnm2.estimates.evaluate_components(images, cnm2.params, dview=dview)

7) %% Plot Evaluated Spatial Components & save plots


In [ ]:
now = datetime.now()  # generate a time stamp
timestamp = str(now.month) + str(now.day) + str(now.year) + '_' + str(now.hour) + str(now.minute) + str(now.second)

if plotting:
    cnm2.estimates.plot_contours(img=Cn, idx=cnm2.estimates.idx_components,cmap='gray')
    if save_results:
        plt.savefig(output_path+os.path.sep+f"{opts.data['animal']}_{opts.data['path'].split(os.path.sep)[-1]}_cnmf_ROIS_Cn_{timestamp}.png")
    if load_prev_mc:
        cnm2.estimates.plot_contours(img=mc_og.total_template_els, idx=cnm2.estimates.idx_components,cmap='gray')
        if save_results:
            plt.savefig(output_path+os.path.sep+f"{opts.data['animal']}_{opts.data['path'].split(os.path.sep)[-1]}_cnmf_ROIS_tpl_{timestamp}.png")

In [ ]:
cnm2.estimates.plot_contours_nb(img=images.mean(axis=0), idx=cnm2.estimates.idx_components,cmap='gray')

In [ ]:
cnm2.estimates.idx_components

# 8) Filter good components, calculate dF/F, save results

In [ ]:
cnm2.estimates.select_components(use_object=True)
    # %%
    # cnm2.estimates.view_components(img=Cn)
    # %% Extract DF/F values
print('Calculating DF/F')
cnm2.estimates.detrend_df_f(quantileMin=8, frames_window=500)

# %% Stop server Save CNMF results
cm.stop_server(dview=dview)
if save_results:
    print('Saving results')
    cnm2.save(output_path+os.path.sep+f"{opts.data['animal']}_{opts.data['path'].split(os.path.sep)[-1]}_cnmfresults_{timestamp}.hdf5")

In [ ]:
cnm2.estimates.nb_view_components(idx=cnm2.estimates.idx_components)


In [ ]:
cnm.estimates.idx_components_bad

9) optional: visualize a certain component

In [ ]:
i=220
cnm2.estimates.view_components(idx=[i],img=Cn)
plt.figure(figsize=(13,4))
plt.plot(cnm2.estimates.F_dff[i,:])

In [ ]:
cnm2.estimates.A.shape

In [ ]:
cnm.estimates.A.shape